# MB2SAP: mouse brain partial-overlap workflow

This notebook applies DVCAlign to the `mMAMP` layout used for two mouse-brain Visium sections: `MA` (sagittal anterior) and `MP` (sagittal posterior). The analysis reproduces the partial-overlap mouse-brain benchmark, including section assembly, DVCAlign training, ARI evaluation against the supplied ground-truth labels, and the MA/MP visualization.

## Data source

Download the expression and spatial files from the official 10x Genomics pages for [Mouse Brain Serial Section 2, Sagittal Anterior](https://www.10xgenomics.com/datasets/mouse-brain-serial-section-2-sagittal-anterior-1-standard) and [Mouse Brain Serial Section 2, Sagittal Posterior](https://www.10xgenomics.com/datasets/mouse-brain-serial-section-2-sagittal-posterior-1-standard). The repository does not redistribute these data. Put the files under `DVCAlign/Data/mMAMP/MA` and `DVCAlign/Data/mMAMP/MP`; optional ground-truth files go in each section's `gt` subdirectory.

The notebook uses the section IDs `MA` and `MP`; these correspond to the anterior and posterior 10x sections above, not to DLPFC, HER2ST, or heart data.

In [ ]:
# Cell 1: imports
import os
from pathlib import Path

_repo_candidates = [Path.cwd(), Path.cwd().parent]
_repo_root = next((p for p in _repo_candidates if (p / 'DVCAlign').is_dir() and (p / 'examples').is_dir()), Path.cwd())
os.chdir(_repo_root)

import warnings
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
from scipy import sparse
from scipy.linalg import block_diag
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
import matplotlib.pyplot as plt
import torch
import DVCAlign

warnings.filterwarnings("ignore")

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMBA_NUM_THREADS"] = "1"
sc.settings.n_jobs = 1

np.random.seed(666)
used_device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")
print("used_device:", used_device)


In [ ]:
root_dir = _repo_root / "DVCAlign" / "Data" / "mMAMP"

section_ids = ["MA", "MP"]

rad_cutoff = 150
n_top_genes = 5000

print("root_dir =", os.path.abspath(root_dir))


In [ ]:
def _sample_values(x, n=5000):
    if sparse.issparse(x):
        arr = x.data
    else:
        arr = np.asarray(x).ravel()
    if arr.size == 0:
        return np.array([0.0], dtype=float)
    if arr.size > n:
        idx = np.random.choice(arr.size, n, replace=False)
        arr = arr[idx]
    return arr.astype(float)

def looks_logged(x):
    s = _sample_values(x)
    frac_non_int = np.mean(np.abs(s - np.round(s)) > 1e-6)
    return (np.nanmax(s) <= 30) and (np.nanpercentile(s, 99) <= 10) and (frac_non_int > 0.2)

def ensure_spatial(adata):
    if "spatial" in adata.obsm:
        sp = adata.obsm["spatial"]
        if isinstance(sp, pd.DataFrame):
            if {"X", "Y"}.issubset(sp.columns):
                adata.obsm["spatial"] = sp[["X", "Y"]].to_numpy()
            elif {"x", "y"}.issubset(sp.columns):
                adata.obsm["spatial"] = sp[["x", "y"]].to_numpy()
            else:
                adata.obsm["spatial"] = np.asarray(sp)
        else:
            adata.obsm["spatial"] = np.asarray(sp)
        return adata


    for cols in [("x4", "x5"), ("x", "y"), ("X", "Y"), ("array_row", "array_col")]:
        if set(cols).issubset(adata.obs.columns):
            adata.obsm["spatial"] = adata.obs[list(cols)].to_numpy()
            return adata

    raise KeyError("error")

def read_gt_labels(root_dir, sid, barcodes):
    gt_candidates = [
        os.path.join(root_dir, sid, "gt", "tissue_positions_list_GTs.txt"),
        os.path.join(root_dir, sid, "gt", "tissue_positions_list_GTs.csv"),
        os.path.join(root_dir, sid, "gt", "ground_truth.csv"),
    ]
    for fp in gt_candidates:
        if not os.path.exists(fp):
            continue

        try:
            gt_df = pd.read_csv(fp, sep=None, engine="python", index_col=0)
        except Exception:
            gt_df = pd.read_csv(fp, index_col=0)


        if "ground_truth" in gt_df.columns:
            col = "ground_truth"
        elif "ground_truth_4" in gt_df.columns:
            col = "ground_truth_4"
        elif "ground_truth_3" in gt_df.columns:
            col = "ground_truth_3"
        elif "ground_truth_2" in gt_df.columns:
            col = "ground_truth_2"
        elif "ground_truth_1" in gt_df.columns:
            col = "ground_truth_1"
        else:
            cand = [c for c in gt_df.columns if ("ground" in c.lower()) or ("cluster" in c.lower())]
            col = cand[0] if len(cand) > 0 else gt_df.columns[-1]

        ser = gt_df[col].astype(str)
        return ser.reindex(barcodes).fillna("unknown").astype(str)

    return pd.Series("unknown", index=barcodes, dtype="object")


In [ ]:

Batch_list, adj_list = [], []

for sid in section_ids:
    print(f"\nLoading {sid} ...")

    candidates = [
        os.path.join(root_dir, sid, f"{sid}1.h5ad"), 
        os.path.join(root_dir, sid, f"{sid}.h5ad"),
        os.path.join(root_dir, sid, f"{sid}_Raw.h5ad"),
        os.path.join(root_dir, sid, f"{sid}_filtered_feature_bc_matrix.h5ad"),
        os.path.join(root_dir, sid, f"{sid}_filtered_feature_bc_matrix.h5"),
    ]
    fn = next((p for p in candidates if os.path.exists(p)), None)
    if fn is None:
        raise FileNotFoundError(f"未找到 {sid} 的表达文件，尝试过: {candidates}")

    if fn.endswith(".h5ad"):
        adata = sc.read_h5ad(fn)
    else:
        adata = sc.read_visium(
            path=os.path.dirname(fn),
            count_file=os.path.basename(fn),
            load_images=False
        )

    adata.var_names_make_unique(join="++")

    if "count" in adata.layers:
        adata.X = adata.layers["count"].copy()

    if not sparse.issparse(adata.X):
        adata.X = sparse.csr_matrix(adata.X)

    adata = ensure_spatial(adata)

    raw_barcodes = adata.obs_names.astype(str)
    gt = read_gt_labels(root_dir, sid, raw_barcodes)
    adata.obs["Ground Truth"] = pd.Categorical(gt)
    adata.obs["batch_name"] = sid

    adata.obs_names = [f"{x}_{sid}" for x in raw_barcodes]

    sc.pp.filter_genes(adata, min_cells=5)
    if not looks_logged(adata.X):
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)

    sc.pp.highly_variable_genes(adata, flavor="seurat", n_top_genes=n_top_genes)
    if "highly_variable" in adata.var.columns and adata.var["highly_variable"].sum() > 0:
        adata = adata[:, adata.var["highly_variable"]].copy()

    DVCAlign.Cal_Spatial_Net(adata, rad_cutoff=rad_cutoff, model="Radius")
    avg_n = adata.uns["Spatial_Net"].shape[0] / adata.n_obs


    Batch_list.append(adata)
    adj_list.append(adata.uns["adj"])
    print(f"[{sid}] shape={adata.shape}, GT classes={adata.obs['Ground Truth'].astype(str).nunique()}")


In [ ]:
adata_concat = ad.concat(Batch_list, label="slice_name", keys=section_ids, uns_merge="same")
adata_concat.obs["batch_name"] = adata_concat.obs["slice_name"].astype("category")
adata_concat.obs["Ground Truth"] = adata_concat.obs["Ground Truth"].astype("category")

if not sparse.issparse(adata_concat.X):
    adata_concat.X = sparse.csr_matrix(adata_concat.X)

adj_concat = np.asarray(adj_list[0].todense())
for i in range(1, len(adj_list)):
    adj_concat = block_diag(adj_concat, np.asarray(adj_list[i].todense()))

adata_concat.uns["edgeList"] = np.nonzero(adj_concat)

print("adata_concat.shape:", adata_concat.shape)
print("edge num:", len(adata_concat.uns["edgeList"][0]))
print("batch_name:", adata_concat.obs["batch_name"].cat.categories.tolist())


In [ ]:

iter_comb = [(0, 1)] 

adata_concat = DVCAlign.train_DVCAlign(
    adata_concat,
    verbose=True,
    knn_neigh=20,
    iter_comb=iter_comb,
    device=used_device,
    aux_candidate_k=30, 
    aux_expr_k=20
)

In [ ]:
num_cluster = 52
DVCAlign.mclust_R(adata_concat, num_cluster=num_cluster, used_obsm="DVCAlign")

valid_mask = ~adata_concat.obs["Ground Truth"].astype(str).isin(["unknown", "NA", "nan", "None"])
adata_eval = adata_concat[valid_mask].copy()

print(f"num_cluster used = {num_cluster}")
print(f"GT valid spots = {adata_eval.n_obs}")

if adata_eval.n_obs > 0:
    ari_all = adjusted_rand_score(adata_eval.obs["Ground Truth"], adata_eval.obs["mclust"])
    nmi_all = normalized_mutual_info_score(adata_eval.obs["Ground Truth"], adata_eval.obs["mclust"])
    print(f"Overall ARI = {ari_all:.4f}")
    print(f"Overall NMI = {nmi_all:.4f}")

for sid in section_ids:
    ad_s = adata_eval[adata_eval.obs["batch_name"] == sid].copy()
    if ad_s.n_obs == 0:
        print(f"{sid}: no valid GT spots")
        continue
    ari_s = adjusted_rand_score(ad_s.obs["Ground Truth"], ad_s.obs["mclust"])
    print(f"{sid}: ARI = {ari_s:.4f}, n_spots = {ad_s.n_obs}")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import to_hex, to_rgb
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import adjusted_rand_score

batch_col = "batch_name"
pred_col = "mclust_plot" if "mclust_plot" in adata_concat.obs.columns else "mclust"
gt_col = "Ground Truth"
gap = 25 


batches = adata_concat.obs[batch_col].astype(str).unique().tolist()

def pick_first(cands, pool):
    return next((x for x in cands if x in pool), None)

sid_ma = pick_first(["MA", "anterior", "Mouse_Brain_Anterior"], batches)
sid_mp = pick_first(["MP", "posterior", "Mouse_Brain_Posterior"], batches)


obs = adata_concat.obs.copy()
ma = obs[obs[batch_col].astype(str) == sid_ma].copy()
mp = obs[obs[batch_col].astype(str) == sid_mp].copy()

def ensure_pixel(df):
    if "x_pixel" not in df.columns or "y_pixel" not in df.columns:
        idx = adata_concat.obs_names.get_indexer(df.index)
        xy = np.asarray(adata_concat.obsm["spatial"])[idx]
        if "x_pixel" not in df.columns: df["x_pixel"] = xy[:, 0]
        if "y_pixel" not in df.columns: df["y_pixel"] = xy[:, 1]
    return df

def pick_col(df, names):
    for c in names:
        if c in df.columns: return c
    return None

def add_pseudo_array(df):
    pts = df[["x_pixel", "y_pixel"]].to_numpy(float)
    if len(pts) < 2:
        pitch = 50.0
    else:
        nn = NearestNeighbors(n_neighbors=2).fit(pts)
        d, _ = nn.kneighbors(pts)
        pitch = float(np.median(d[:, 1]))
        if not np.isfinite(pitch) or pitch <= 0: pitch = 50.0
    out = df.copy()
    out["_x_array"] = np.round((out["x_pixel"] - out["x_pixel"].min()) / pitch).astype(int)
    out["_y_array"] = np.round((out["y_pixel"] - out["y_pixel"].min()) / pitch).astype(int)
    return out

ma, mp = ensure_pixel(ma), ensure_pixel(mp)
xarr_col = pick_col(ma, ["x_array", "array_row"])
yarr_col = pick_col(ma, ["y_array", "array_col"])
if xarr_col is None or yarr_col is None:
    ma, mp = add_pseudo_array(ma), add_pseudo_array(mp)
    xarr_col, yarr_col = "_x_array", "_y_array"

ma["x_pixel_aligned"], ma["y_pixel_aligned"] = ma["x_pixel"].astype(float), ma["y_pixel"].astype(float)
mp["x_pixel_aligned"], mp["y_pixel_aligned"] = mp["x_pixel"].astype(float), mp["y_pixel"].astype(float)

# 复刻 Run_MBA_MBP 拼接
mp["y_pixel_aligned"] = mp["y_pixel_aligned"] - mp["y_pixel_aligned"].min() + ma["y_pixel_aligned"].max() + gap
ma["boundary"], mp["boundary"] = 0, 0

for i in np.unique(ma[xarr_col]):
    t = ma[ma[xarr_col] == i]
    ma.loc[t[t[yarr_col] == t[yarr_col].max()].index, "boundary"] = 1

for i in np.unique(mp[xarr_col]):
    t = mp[mp[xarr_col] == i]
    mp.loc[t[t[yarr_col] == t[yarr_col].min()].index, "boundary"] = 1

mp["x_pixel_aligned"] += ma.loc[ma["boundary"] == 1, "x_pixel_aligned"].mean() - mp.loc[mp["boundary"] == 1, "x_pixel_aligned"].mean()

plot_df = pd.concat([ma, mp], axis=0).copy()
plot_df[pred_col] = plot_df[pred_col].astype(str)

def sort_key(x):
    s = str(x)
    return (0, int(s)) if s.isdigit() else (1, s)

def _mix_toward(color, target, weight):
    base = np.asarray(to_rgb(color))
    target = np.asarray(target, dtype=float)
    return to_hex(np.clip(base * (1 - weight) + target * weight, 0, 1))

def _build_model_palette(n):
    base_colors = [
        "#5B8FF9", "#45B7D1", "#5BC49F", "#9CCC65",
        "#D4B96E", "#F3A45B", "#E67E5F", "#D16D8C",
        "#B07CC6", "#7E8CE0", "#76C4E2", "#4DB6AC",
    ]
    palette = []
    for weight in (0.00, 0.18, 0.34):
        palette.extend(_mix_toward(color, (1, 1, 1), weight) for color in base_colors)
    palette.extend(_mix_toward(color, (0, 0, 0), 0.12) for color in base_colors)
    palette.extend(_mix_toward(color, (1, 1, 1), 0.48) for color in base_colors)
    repeats = int(np.ceil(n / len(palette)))
    return (palette * max(1, repeats))[:n]

clusters = sorted(plot_df[pred_col].dropna().unique(), key=sort_key)
palette = _build_model_palette(max(3, len(clusters)))
pal = {k: palette[i] for i, k in enumerate(clusters)}

def calc_ari(df):
    if gt_col not in df.columns: return "NA"
    gt = df[gt_col].astype(str)
    valid = ~gt.isin(["unknown", "NA", "nan", "None"])
    return f"{adjusted_rand_score(gt[valid], df.loc[valid, pred_col].astype(str)):.3f}" if valid.any() else "NA"

ari_ma = calc_ari(plot_df[plot_df[batch_col].astype(str) == sid_ma])
ari_mp = calc_ari(plot_df[plot_df[batch_col].astype(str) == sid_mp])

plt.rcParams["axes.facecolor"] = "white"
plt.rcParams["figure.facecolor"] = "white"
fig, ax = plt.subplots(figsize=(14, 5.2), dpi=180)

ax.scatter(
    plot_df["y_pixel_aligned"].values,
    plot_df["x_pixel_aligned"].values,
    c=plot_df[pred_col].map(pal).fillna("#BDBDBD").values,
    s=200000 / max(1, plot_df.shape[0]),
    linewidths=0, alpha=1.0, rasterized=True
)

ax.set_aspect("equal", "box")
ax.invert_yaxis()
ax.grid(False)


ax.set_title("")
ax.set_xlabel("")
ax.set_ylabel("")
ax.axis("off")


t_ma = plot_df[plot_df[batch_col].astype(str) == sid_ma]
t_mp = plot_df[plot_df[batch_col].astype(str) == sid_mp]


x_mid = 0.5 * (plot_df["y_pixel_aligned"].min() + plot_df["y_pixel_aligned"].max())
y_mid = 0.5 * (plot_df["x_pixel_aligned"].min() + plot_df["x_pixel_aligned"].max())
ax.text(0.5, 1.02, f"ARI={ari_ma}",
        transform=ax.transAxes, ha="center", va="bottom",
        fontsize=13, fontweight="bold", color="black", clip_on=False)

handles = [
    Line2D([0], [0], marker='o', color='w', label=str(k),
           markerfacecolor=pal[k], markersize=7)
    for k in clusters
]
ax.legend(handles=handles, ncol=3, frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.show()
